# Word2Vec — When Words Become Geometry

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Notebook Guide (connected to Lab 43: The Memory Palace)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/appendix-3.ipynb)

**What you will do:** recombine different word vectors from the toy embedding table, and add noise to them, to see when meaning-as-geometry arithmetic works and when it breaks down.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Simulation

**Concept:** Your brain represents the meaning of words as patterns of neural activity—and similar meanings produce similar patterns ([→ Lab 43](#lab-43), semantic memory). AI does something closely analogous: it maps words to vectors in a high-dimensional space where meaning becomes geometry.

**Relevant Labs:** Lab 35 (semantic satiation), Lab 43 (memory palace / spatial encoding of memory)

In [ ]:
# Word2Vec: Meaning as Geometry

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# This demonstration uses small, hand-crafted vectors so the geometry
# is easy to follow. A real Word2Vec model (installable in Colab with
# "!pip install gensim") learns vectors like these automatically from
# millions of sentences, but the underlying idea is the same.
words = {
    'king': [0.9, 0.8, 0.1, 0.2],
    'queen': [0.9, 0.8, 0.9, 0.2],
    'man': [0.5, 0.5, 0.1, 0.5],
    'woman': [0.5, 0.5, 0.9, 0.5],
    'prince': [0.7, 0.6, 0.1, 0.3],
    'princess': [0.7, 0.6, 0.9, 0.3],
    'boy': [0.3, 0.3, 0.1, 0.6],
    'girl': [0.3, 0.3, 0.9, 0.6],
    'dog': [0.1, 0.1, 0.5, 0.9],
    'cat': [0.1, 0.2, 0.5, 0.8],
    'paris': [0.8, 0.1, 0.5, 0.1],
    'france': [0.8, 0.2, 0.5, 0.1],
    'tokyo': [0.7, 0.1, 0.5, 0.15],
    'japan': [0.7, 0.2, 0.5, 0.15],
}

names = list(words.keys())
vectors = np.array(list(words.values()))

# PCA to 2D for visualisation
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: word map ---
colours = {
    'king': 'gold', 'queen': 'gold',
    'man': 'steelblue', 'woman': 'salmon',
    'prince': 'gold', 'princess': 'gold',
    'boy': 'steelblue', 'girl': 'salmon',
    'dog': 'green', 'cat': 'green',
    'paris': 'purple', 'france': 'purple',
    'tokyo': 'purple', 'japan': 'purple',
}

for i, name in enumerate(names):
    ax1.scatter(coords[i, 0], coords[i, 1], c=colours[name], s=100,
                edgecolors='black', linewidth=0.5, zorder=5)
    ax1.annotate(name, (coords[i, 0], coords[i, 1]),
                 textcoords="offset points", xytext=(8, 8), fontsize=11)

# Draw the famous analogy arrow: king - man + woman ≈ queen
king_v = words['king']
man_v = words['man']
woman_v = words['woman']
result = np.array(king_v) - np.array(man_v) + np.array(woman_v)
result_2d = pca.transform(result.reshape(1, -1))[0]

king_2d = coords[names.index('king')]

ax1.annotate('', xy=result_2d, xytext=king_2d,
             arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax1.scatter(*result_2d, c='red', s=150, marker='*', zorder=6,
            label='king - man + woman = ?')

ax1.set_title('Words as Points in Space\n(king − man + woman ≈ queen)',
              fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Right: analogy arithmetic ---
analogies = [
    ('king', 'man', 'woman', 'queen'),
    ('paris', 'france', 'japan', 'tokyo'),
    ('boy', 'girl', 'man', 'woman'),
]

y_positions = [3, 2, 1]
for idx, (a, b, c, expected) in enumerate(analogies):
    vec_a = np.array(words[a])
    vec_b = np.array(words[b])
    vec_c = np.array(words[c])
    result_vec = vec_a - vec_b + vec_c

    # Find the closest word to the result (excluding the inputs)
    best_word = None
    best_dist = float('inf')
    for name, vec in words.items():
        if name in [a, b, c]:
            continue
        dist = np.linalg.norm(result_vec - np.array(vec))
        if dist < best_dist:
            best_dist = dist
            best_word = name

    text = f"{a} − {b} + {c} = {best_word}"
    colour = 'green' if best_word == expected else 'orange'
    ax2.text(0.5, y_positions[idx], text, fontsize=14, ha='center',
             va='center', fontweight='bold', color=colour,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                       edgecolor=colour, linewidth=2))

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 4)
ax2.set_title('Word Arithmetic:\nMeaning Has Direction', fontsize=13)
ax2.axis('off')

plt.tight_layout()
plt.savefig('word2vec_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: in both brains and AI, meaning is represented as")
print("patterns in a high-dimensional space. Similar meanings cluster")
print("together, and relationships between meanings become geometric")
print("operations—just as 'king minus man plus woman equals queen.'")
print("Not every combination lines up this neatly (see the orange")
print("result on the right): real word-vector spaces are approximate,")
print("not exact arithmetic.")

## Change the parameters

Pick any three words already in the table for **a**, **b** and **c** to compute
`a - b + c`, and use **noise** to blur every word's vector a little, simulating a less
precisely learned embedding space. The widget reuses the same `words` dictionary,
`colours` map and PCA projection as the cell above.
(If the widgets do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, Dropdown, FloatSlider

def explore(a="king", b="man", c="woman", noise=0.0):
    rng = np.random.RandomState(0)
    noisy_words = {k: np.array(v) + rng.normal(0, noise, 4) for k, v in words.items()}
    vec_a, vec_b, vec_c = noisy_words[a], noisy_words[b], noisy_words[c]
    result_vec = vec_a - vec_b + vec_c

    best_word, best_dist = None, float("inf")
    for name, vec in noisy_words.items():
        if name in (a, b, c):
            continue
        dist = np.linalg.norm(result_vec - vec)
        if dist < best_dist:
            best_dist, best_word = dist, name
    print(f"{a} - {b} + {c} = {best_word}  (distance {best_dist:.3f})")

    vectors_local = np.array(list(noisy_words.values()))
    names_local = list(noisy_words.keys())
    pca_local = PCA(n_components=2)
    coords_local = pca_local.fit_transform(vectors_local)

    fig, ax = plt.subplots(figsize=(7, 6))
    for i, name in enumerate(names_local):
        ax.scatter(*coords_local[i], c=colours.get(name, "grey"), s=80, edgecolors="black", linewidth=0.5, zorder=5)
        ax.annotate(name, coords_local[i], textcoords="offset points", xytext=(6, 6), fontsize=9)
    result_2d = pca_local.transform(result_vec.reshape(1, -1))[0]
    a_2d = coords_local[names_local.index(a)]
    ax.annotate("", xy=result_2d, xytext=a_2d, arrowprops=dict(arrowstyle="->", color="red", lw=2))
    ax.scatter(*result_2d, c="red", s=150, marker="*", zorder=6, label=f"{a} - {b} + {c}")
    ax.set_title(f"noise = {noise}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

names = list(words.keys())
interact(explore,
         a=Dropdown(options=names, value="king"),
         b=Dropdown(options=names, value="man"),
         c=Dropdown(options=names, value="woman"),
         noise=FloatSlider(value=0.0, min=0.0, max=0.5, step=0.02));

## Questions to think about

1. Try boy minus girl plus man. Does the nearest word correctly come out as "woman"? Now try paris minus france plus japan. Both are structurally the same kind of relationship -- does one analogy work better than the other in this small toy embedding?
2. Slowly raise noise from 0 toward 0.3, rerunning king minus man plus woman each time. At roughly what noise level does the nearest word stop being "queen"? What does this tell you about how much an embedding's directions can be perturbed before "meaning as geometry" breaks down?
3. Try an analogy that mixes unrelated categories, such as dog minus cat plus paris. What word comes out, and does the result mean anything sensible? What does this say about the limits of vector arithmetic when the two paired concepts do not share the same kind of relationship?
4. Lab 43 explains that the hippocampus's place and grid cells create a spatial map that memory athletes use for the method of loci, while embeddings create a mathematical map that AI uses. Looking at your PCA plot, do semantically related words such as king, queen, prince and princess cluster together the way locations on a memory-palace route would need to for that technique to work?

## Challenge

Add one or two new words of your own to the `words` dictionary with hand-picked
four-dimensional vectors, and see whether you can construct an analogy that works with them
-- much as a person designing their own memory-palace route has to choose locations that
actually support the associations they want to recall.

In [ ]:
# Example data: two new hand-crafted words extending the toy vocabulary.
# The four numbers are made up to test whether they behave analogously to an existing
# category -- adjust them and see what changes.
my_words = dict(words)  # start from the original 14-word vectors
my_words["actor"] = [0.6, 0.7, 0.1, 0.4]     # example data
my_words["actress"] = [0.6, 0.7, 0.9, 0.4]   # example data

def nearest(result_vec, vocab, exclude):
    best_word, best_dist = None, float("inf")
    for name, vec in vocab.items():
        if name in exclude:
            continue
        dist = np.linalg.norm(result_vec - np.array(vec))
        if dist < best_dist:
            best_dist, best_word = dist, name
    return best_word, best_dist

result = np.array(my_words["actor"]) - np.array(my_words["man"]) + np.array(my_words["woman"])
best_word, best_dist = nearest(result, my_words, ["actor", "man", "woman"])
print(f"actor - man + woman = {best_word} (distance {best_dist:.3f})")
print("The 'actor'/'actress' vectors above are example data made up by hand; a real Word2Vec")
print("model would learn such vectors automatically from a large text corpus instead.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "appendix-3.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")